In [ ]:
import warnings, os
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import (RandomForestRegressor, RandomForestClassifier,
                               GradientBoostingRegressor, GradientBoostingClassifier)
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                              classification_report, roc_auc_score, roc_curve,
                              precision_score, recall_score, f1_score)
import xgboost as xgb
import shap

import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (LSTM, GRU, Dense, Dropout, Bidirectional,
                                      Conv1D, MaxPooling1D, Input,
                                      MultiHeadAttention, LayerNormalization,
                                      GlobalAveragePooling1D)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

np.random.seed(42)
tf.random.set_seed(42)


In [ ]:
PATHS = {
    "ds1": "data/ds1_fci_wheat_procurement_2000_2023.csv",
    "ds2": "data/ds2_imd_meteorological_1990_2023.csv",
    "ds3": "data/ds3_fao_agricultural_commodity_1990_2022.csv",
    "ds4": "data/ds4_fci_storage_transportation_2005_2023.csv",
}

ds1 = pd.read_csv(PATHS["ds1"])
ds2 = pd.read_csv(PATHS["ds2"])
ds3 = pd.read_csv(PATHS["ds3"])
ds4 = pd.read_csv(PATHS["ds4"])

ds3 = ds3[(ds3["Country"] == "India") & (ds3["Commodity"] == "Wheat")].copy()


In [ ]:
ds1.dropna(subset=["Year","Month","State","Quantity_MT"], inplace=True)
ds1["Year"]  = ds1["Year"].astype(int)
ds1["Month"] = ds1["Month"].astype(int)
ds1["State"] = ds1["State"].str.strip().str.title()

ds2.dropna(subset=["Year","Month","State"], inplace=True)
ds2["State"] = ds2["State"].str.strip().str.title()
ds2_state = (ds2.groupby(["Year","Month","State"])
               .agg(Rainfall_mm   = ("Rainfall_mm","mean"),
                    Temperature_C = ("Temperature_C","mean"),
                    Humidity_pct  = ("Humidity_pct","mean"),
                    Drought_Index = ("Drought_Index","mean"))
               .reset_index())

ds3.dropna(subset=["Year","Production_MT"], inplace=True)
ds3["Year"] = ds3["Year"].astype(int)
ds3_annual = ds3[["Year","Production_MT","Import_MT","Export_MT",
                   "Domestic_Supply_MT","Population"]].copy()

ds4.dropna(subset=["Year","Quarter","State"], inplace=True)
ds4["State"] = ds4["State"].str.strip().str.title()
ds4["Year"]  = ds4["Year"].astype(int)

df_rq1 = pd.merge(ds1, ds2_state, on=["Year","Month","State"], how="inner")
df_rq1 = pd.merge(df_rq1, ds3_annual, on="Year", how="left")
df_rq1 = df_rq1[(df_rq1["Year"] >= 2005) & (df_rq1["Year"] <= 2022)].copy()
df_rq1.sort_values(["State","Year","Month"], inplace=True)
df_rq1.reset_index(drop=True, inplace=True)


In [ ]:
ds1["Quarter"] = ((ds1["Month"] - 1) // 3) + 1
df_rq2 = pd.merge(
    ds1.groupby(["Year","Quarter","State"])
       .agg(Quantity_MT          = ("Quantity_MT","sum"),
            MSP                  = ("MSP","mean"),
            Storage_Capacity_Pct = ("Storage_Capacity_Pct","mean"))
       .reset_index(),
    ds4, on=["Year","Quarter","State"], how="inner"
)
df_rq2 = df_rq2[(df_rq2["Year"] >= 2005) & (df_rq2["Year"] <= 2022)].copy()
df_rq2.reset_index(drop=True, inplace=True)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

annual = df_rq1.groupby("Year")["Quantity_MT"].sum() / 1e6
axes[0].plot(annual.index, annual.values, marker="o", color="#2563eb", linewidth=2)
axes[0].set(title="Annual Wheat Procurement (MT millions)", xlabel="Year", ylabel="MT (millions)")
axes[0].grid(alpha=0.3)

monthly = df_rq1.groupby("Month")["Quantity_MT"].mean() / 1e3
axes[1].bar(monthly.index, monthly.values, color="#16a34a")
axes[1].set(title="Avg Monthly Procurement (MT thousands)", xlabel="Month", ylabel="MT (thousands)")
axes[1].grid(alpha=0.3, axis="y")

top_states = df_rq1.groupby("State")["Quantity_MT"].sum().nlargest(10) / 1e6
axes[2].barh(top_states.index[::-1], top_states.values[::-1], color="#dc2626")
axes[2].set(title="Top 10 States by Procurement Volume", xlabel="MT (millions)")
axes[2].grid(alpha=0.3, axis="x")

plt.tight_layout()
plt.savefig("results/eda_overview.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
FEATURE_COLS = ["Quantity_MT","MSP","Rainfall_mm","Temperature_C",
                "Humidity_pct","Drought_Index","Month"]
TARGET_COL   = "Quantity_MT"
LOOKBACK     = 12
HORIZON      = 3

le = LabelEncoder()
df_rq1["State_enc"] = le.fit_transform(df_rq1["State"])
FEATURE_COLS_ENC = FEATURE_COLS + ["State_enc"]

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
df_rq1[FEATURE_COLS_ENC] = scaler_X.fit_transform(df_rq1[FEATURE_COLS_ENC])
df_rq1[[TARGET_COL]]      = scaler_y.fit_transform(df_rq1[[TARGET_COL]])

def make_sequences(data, feature_cols, target_col, lookback, horizon):
    X, y = [], []
    arr  = data[feature_cols].values
    tgt  = data[target_col].values
    for i in range(lookback, len(arr) - horizon + 1):
        X.append(arr[i - lookback : i])
        y.append(tgt[i : i + horizon])
    return np.array(X), np.array(y)

X_all, y_all = [], []
for state in df_rq1["State"].unique():
    sub = df_rq1[df_rq1["State"] == state].copy()
    if len(sub) > LOOKBACK + HORIZON:
        Xs, ys = make_sequences(sub, FEATURE_COLS_ENC, TARGET_COL, LOOKBACK, HORIZON)
        X_all.append(Xs)
        y_all.append(ys)

X_all = np.concatenate(X_all, axis=0)
y_all = np.concatenate(y_all, axis=0)

n      = len(X_all)
t1, t2 = int(0.70*n), int(0.85*n)
X_train, X_val, X_test = X_all[:t1], X_all[t1:t2], X_all[t2:]
y_train, y_val, y_test = y_all[:t1], y_all[t1:t2], y_all[t2:]


In [ ]:
n_features = X_train.shape[2]
ES = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

def build_lstm():
    m = Sequential([
        LSTM(128, return_sequences=True, input_shape=(LOOKBACK, n_features)),
        Dropout(0.2),
        LSTM(64),
        Dropout(0.2),
        Dense(HORIZON)
    ])
    m.compile(Adam(1e-3), "mse")
    return m

def build_gru():
    m = Sequential([
        GRU(128, return_sequences=True, input_shape=(LOOKBACK, n_features)),
        Dropout(0.2),
        GRU(64),
        Dropout(0.2),
        Dense(HORIZON)
    ])
    m.compile(Adam(1e-3), "mse")
    return m

def build_bilstm():
    m = Sequential([
        Bidirectional(LSTM(128, return_sequences=True), input_shape=(LOOKBACK, n_features)),
        Dropout(0.2),
        Bidirectional(LSTM(64)),
        Dropout(0.2),
        Dense(HORIZON)
    ])
    m.compile(Adam(1e-3), "mse")
    return m

def build_cnn_lstm():
    m = Sequential([
        Conv1D(64, kernel_size=3, activation="relu", input_shape=(LOOKBACK, n_features)),
        MaxPooling1D(pool_size=2),
        LSTM(64),
        Dropout(0.2),
        Dense(HORIZON)
    ])
    m.compile(Adam(1e-3), "mse")
    return m

def build_tft():
    inp = Input(shape=(LOOKBACK, n_features))
    x   = Dense(64, activation="relu")(inp)
    x, _ = MultiHeadAttention(num_heads=4, key_dim=16)(x, x, return_attention_scores=True)
    x   = LayerNormalization()(x)
    x   = GlobalAveragePooling1D()(x)
    x   = Dense(64, activation="relu")(x)
    x   = Dropout(0.2)(x)
    out = Dense(HORIZON)(x)
    m   = Model(inp, out)
    m.compile(Adam(1e-3), "mse")
    return m

MODELS = {
    "LSTM"     : build_lstm,
    "GRU"      : build_gru,
    "BiLSTM"   : build_bilstm,
    "CNN-LSTM" : build_cnn_lstm,
    "TFT"      : build_tft,
}


In [ ]:
def mape(y_true, y_pred):
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

rq1_results   = {}
trained_models = {}

for name, builder in MODELS.items():
    model = builder()
    model.fit(X_train, y_train,
              validation_data=(X_val, y_val),
              epochs=200, batch_size=32,
              callbacks=[ES], verbose=0)

    pred        = model.predict(X_test, verbose=0)
    y_true_inv  = scaler_y.inverse_transform(y_test[:, :1]).flatten()
    y_pred_inv  = scaler_y.inverse_transform(pred[:, :1]).flatten()

    rq1_results[name] = {
        "RMSE" : np.sqrt(mean_squared_error(y_true_inv, y_pred_inv)),
        "MAE"  : mean_absolute_error(y_true_inv, y_pred_inv),
        "MAPE%": mape(y_true_inv, y_pred_inv),
        "R2"   : r2_score(y_true_inv, y_pred_inv)
    }
    trained_models[name] = model

from statsmodels.tsa.arima.model import ARIMA
try:
    sample = df_rq1[df_rq1["State"] == df_rq1["State"].unique()[0]]["Quantity_MT"].values
    split  = int(0.7 * len(sample))
    arima  = ARIMA(sample[:split], order=(2,1,2)).fit()
    fc     = arima.forecast(steps=len(sample)-split)
    at     = scaler_y.inverse_transform(sample[split:].reshape(-1,1)).flatten()
    ap     = scaler_y.inverse_transform(np.array(fc).reshape(-1,1)).flatten()
    rq1_results["ARIMA (Baseline)"] = {
        "RMSE" : np.sqrt(mean_squared_error(at, ap)),
        "MAE"  : mean_absolute_error(at, ap),
        "MAPE%": mape(at, ap),
        "R2"   : r2_score(at, ap)
    }
except Exception as e:
    print("ARIMA skipped:", e)

df_rq1_results = pd.DataFrame(rq1_results).T.sort_values("MAPE%")
display(df_rq1_results.round(3))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

order  = df_rq1_results.sort_values("MAPE%", ascending=False)
colors = ["#dc2626" if "ARIMA" in i else "#2563eb" for i in order.index]
axes[0].barh(order.index, order["MAPE%"], color=colors)
axes[0].set(title="RQ1 — MAPE% by Model", xlabel="MAPE (%)")
axes[0].grid(alpha=0.3, axis="x")

order2  = df_rq1_results.sort_values("R2")
colors2 = ["#dc2626" if "ARIMA" in i else "#16a34a" for i in order2.index]
axes[1].barh(order2.index, order2["R2"], color=colors2)
axes[1].set(title="RQ1 — R² by Model", xlabel="R²")
axes[1].grid(alpha=0.3, axis="x")

plt.tight_layout()
plt.savefig("results/rq1_model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

best_model_name = df_rq1_results["MAPE%"].idxmin()
best_model      = trained_models[best_model_name]
pred_best       = best_model.predict(X_test, verbose=0)

y_true_plot = scaler_y.inverse_transform(y_test[:, :1]).flatten()[:100]
y_pred_plot = scaler_y.inverse_transform(pred_best[:, :1]).flatten()[:100]

plt.figure(figsize=(14, 4))
plt.plot(y_true_plot, label="Actual", color="#1e293b", linewidth=1.5)
plt.plot(y_pred_plot, label=f"{best_model_name} Predicted",
         color="#2563eb", linestyle="--", linewidth=1.5)
plt.title(f"RQ1 — Actual vs Predicted ({best_model_name})")
plt.xlabel("Sample"); plt.ylabel("Quantity (MT)")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("results/rq1_actual_vs_predicted.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
df_opt = df_rq2.dropna().copy()
for col in df_opt.select_dtypes(include="object").columns:
    df_opt[col] = LabelEncoder().fit_transform(df_opt[col].astype(str))


In [ ]:
PRICE_FEATURES = [c for c in df_opt.columns
                  if c not in ["MSP","Transport_Cost_INR_MT","Offtake_MT"]]

X_p = df_opt[PRICE_FEATURES]
y_p = df_opt["MSP"]
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(X_p, y_p, test_size=0.2, random_state=42)

rf_price = RandomForestRegressor(n_estimators=300, max_depth=15,
                                  min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_price.fit(Xp_tr, yp_tr)
yp_pred = rf_price.predict(Xp_te)

rf_price_metrics = {
    "RMSE" : np.sqrt(mean_squared_error(yp_te, yp_pred)),
    "MAE"  : mean_absolute_error(yp_te, yp_pred),
    "MAPE%": np.mean(np.abs((yp_te - yp_pred) / yp_te.replace(0, np.nan))) * 100,
    "R2"   : r2_score(yp_te, yp_pred)
}
print(rf_price_metrics)

fi_rf = pd.Series(rf_price.feature_importances_, index=PRICE_FEATURES).nlargest(10)
plt.figure(figsize=(10, 4))
fi_rf.sort_values().plot(kind="barh", color="#2563eb")
plt.title("RF Feature Importance — Procurement Price")
plt.tight_layout()
plt.savefig("results/rq2_rf_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
STORAGE_FEATURES = [c for c in df_opt.columns
                    if c not in ["Stock_Available_MT","Offtake_MT","Transport_Cost_INR_MT"]]

X_s = df_opt[STORAGE_FEATURES]
y_s = df_opt["Stock_Available_MT"]
Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(X_s, y_s, test_size=0.2, random_state=42)

lr_s = LinearRegression().fit(Xs_tr, ys_tr)
rf_s = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1).fit(Xs_tr, ys_tr)
xgb_s = xgb.XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.05,
                           subsample=0.8, colsample_bytree=0.8,
                           random_state=42, n_jobs=-1, verbosity=0)
xgb_s.fit(Xs_tr, ys_tr, eval_set=[(Xs_te, ys_te)], verbose=False)

storage_results = {}
for label, model in [("Linear Regression", lr_s), ("Random Forest", rf_s), ("XGBoost", xgb_s)]:
    pred = model.predict(Xs_te)
    storage_results[label] = {
        "MAE"              : mean_absolute_error(ys_te, pred),
        "R2"               : r2_score(ys_te, pred),
        "Surplus_Waste_pct": np.mean(np.maximum(pred - ys_te, 0) / (ys_te + 1)) * 100
    }

df_storage = pd.DataFrame(storage_results).T
display(df_storage.round(3))
df_storage.to_csv("results/rq2_storage_allocation.csv")


In [ ]:
SCHED_FEATURES = [c for c in df_opt.columns
                  if c not in ["Quantity_MT","Offtake_MT"]]

X_q = df_opt[SCHED_FEATURES]
y_q = df_opt["Quantity_MT"]
Xq_tr, Xq_te, yq_tr, yq_te = train_test_split(X_q, y_q, test_size=0.2, random_state=42)

gbm_sched = GradientBoostingRegressor(n_estimators=400, max_depth=4,
                                       learning_rate=0.05, subsample=0.75,
                                       min_samples_leaf=10, random_state=42)
gbm_sched.fit(Xq_tr, yq_tr)
yq_pred = gbm_sched.predict(Xq_te)

gbm_mape = np.mean(np.abs((yq_te - yq_pred) / yq_te.replace(0, np.nan))) * 100
gbm_r2   = r2_score(yq_te, yq_pred)
print(f"GBM Procurement Scheduler — MAPE: {gbm_mape:.2f}%  R²: {gbm_r2:.3f}")

over_proc  = np.mean(yq_pred > yq_te * 1.05) * 100
under_proc = np.mean(yq_pred < yq_te * 0.95) * 100
print(f"Over-procurement : {over_proc:.1f}%")
print(f"Under-procurement: {under_proc:.1f}%")


In [ ]:
TRANSPORT_FEATURES = [c for c in df_opt.columns
                      if c not in ["Transport_Cost_INR_MT","Offtake_MT"]]

X_t = df_opt[TRANSPORT_FEATURES]
y_t = df_opt["Transport_Cost_INR_MT"]
Xt_tr, Xt_te, yt_tr, yt_te = train_test_split(X_t, y_t, test_size=0.2, random_state=42)

sc_t     = MinMaxScaler()
Xt_tr_sc = sc_t.fit_transform(Xt_tr)
Xt_te_sc = sc_t.transform(Xt_te)

mlp = MLPRegressor(hidden_layer_sizes=(256, 128, 64), activation="relu",
                   solver="adam", learning_rate_init=1e-3, max_iter=500,
                   early_stopping=True, validation_fraction=0.15, random_state=42)
mlp.fit(Xt_tr_sc, yt_tr)
yt_pred = mlp.predict(Xt_te_sc)

mlp_rmse = np.sqrt(mean_squared_error(yt_te, yt_pred))
mlp_mape = np.mean(np.abs((yt_te - yt_pred) / yt_te.replace(0, np.nan))) * 100
print(f"MLP Transport Cost — RMSE: {mlp_rmse:.2f}  MAPE: {mlp_mape:.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].scatter(yq_te[:300], yq_pred[:300], alpha=0.4, color="#2563eb", s=10)
axes[0].plot([yq_te.min(), yq_te.max()], [yq_te.min(), yq_te.max()], "r--")
axes[0].set(title="GBM Procurement Scheduling — Actual vs Predicted",
            xlabel="Actual (MT)", ylabel="Predicted")
axes[0].grid(alpha=0.3)

axes[1].scatter(yt_te[:300], yt_pred[:300], alpha=0.4, color="#16a34a", s=10)
axes[1].plot([yt_te.min(), yt_te.max()], [yt_te.min(), yt_te.max()], "r--")
axes[1].set(title="MLP Transport Cost — Actual vs Predicted",
            xlabel="Actual (INR/MT)", ylabel="Predicted")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("results/rq2_optimization_scatter.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
df_dss = df_opt.copy()
df_dss["Demand_Supply_Gap"] = (df_dss["Quantity_MT"] - df_dss["Stock_Available_MT"]) / (df_dss["Stock_Available_MT"] + 1)
df_dss["Storage_Util"]      = df_dss["Stock_Available_MT"] / (df_dss["Storage_Capacity_MT"] + 1)
df_dss["High_Risk"]         = ((df_dss["Demand_Supply_Gap"] > 0.30) | (df_dss["Storage_Util"] > 0.90)).astype(int)

RISK_FEATURES = [c for c in df_dss.columns
                 if c not in ["High_Risk","Demand_Supply_Gap"]]

X_r = df_dss[RISK_FEATURES]
y_r = df_dss["High_Risk"]
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(X_r, y_r, test_size=0.2,
                                                random_state=42, stratify=y_r)

rf_risk = RandomForestClassifier(n_estimators=300, max_depth=15,
                                  class_weight="balanced", random_state=42, n_jobs=-1)
rf_risk.fit(Xr_tr, yr_tr)
yr_pred = rf_risk.predict(Xr_te)
yr_prob = rf_risk.predict_proba(Xr_te)[:, 1]
auc_roc = roc_auc_score(yr_te, yr_prob)

print(f"AUC-ROC: {auc_roc:.3f}")
print(classification_report(yr_te, yr_pred, target_names=["Low Risk","High Risk"]))

fpr, tpr, _ = roc_curve(yr_te, yr_prob)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color="#2563eb", linewidth=2, label=f"RF Classifier (AUC={auc_roc:.3f})")
plt.plot([0,1],[0,1],"k--", linewidth=1)
plt.fill_between(fpr, tpr, alpha=0.1, color="#2563eb")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("RQ3 — Distribution Risk Classifier ROC Curve")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("results/rq3_roc_curve.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
df_dss["Offtake_roll30"] = df_dss["Offtake_MT"].rolling(3, min_periods=1).mean()
df_dss["Anomaly"]        = (df_dss["Offtake_MT"] < df_dss["Offtake_roll30"] * 0.60).astype(int)

ANOM_FEATURES = [c for c in df_dss.columns
                 if c not in ["High_Risk","Anomaly","Offtake_MT","Demand_Supply_Gap",
                               "Storage_Util","Offtake_roll30"]]

X_a = df_dss[ANOM_FEATURES]
y_a = df_dss["Anomaly"]
Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(X_a, y_a, test_size=0.2,
                                                random_state=42, stratify=y_a)

gbm_anom = GradientBoostingClassifier(n_estimators=400, max_depth=4,
                                       learning_rate=0.05, subsample=0.75,
                                       random_state=42)
gbm_anom.fit(Xa_tr, ya_tr)
ya_pred = gbm_anom.predict(Xa_te)

prec = precision_score(ya_te, ya_pred, zero_division=0)
rec  = recall_score(ya_te, ya_pred, zero_division=0)
f1   = f1_score(ya_te, ya_pred, zero_division=0)
print(f"Anomaly Detector — Precision: {prec:.3f}  Recall: {rec:.3f}  F1: {f1:.3f}")

fi_anom = pd.Series(gbm_anom.feature_importances_, index=ANOM_FEATURES).nlargest(10)
plt.figure(figsize=(10, 4))
fi_anom.sort_values().plot(kind="barh", color="#dc2626")
plt.title("GBM Feature Importance — Anomaly Detection")
plt.tight_layout()
plt.savefig("results/rq3_anomaly_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
explainer = shap.TreeExplainer(rf_risk)
shap_vals = explainer.shap_values(Xr_te.iloc[:200])
sv        = shap_vals[1] if isinstance(shap_vals, list) else shap_vals

plt.figure(figsize=(12, 6))
shap.summary_plot(sv, Xr_te.iloc[:200], feature_names=RISK_FEATURES,
                  plot_type="bar", show=False)
plt.title("SHAP — Global Feature Importance (Risk Classifier)")
plt.tight_layout()
plt.savefig("results/rq3_shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

high_risk_idx = np.where(yr_te.values == 1)[0]
if len(high_risk_idx) > 0:
    idx = high_risk_idx[0]
    shap_exp = shap.Explanation(
        values      = sv[idx],
        base_values = (explainer.expected_value[1]
                       if isinstance(explainer.expected_value, list)
                       else explainer.expected_value),
        data          = Xr_te.iloc[idx].values,
        feature_names = RISK_FEATURES
    )
    plt.figure(figsize=(10, 5))
    shap.waterfall_plot(shap_exp, show=False)
    plt.title("SHAP Waterfall — Single High-Risk District Alert")
    plt.tight_layout()
    plt.savefig("results/rq3_shap_waterfall.png", dpi=150, bbox_inches="tight")
    plt.show()


In [ ]:
dss_input = X_test[-30:]
dss_preds = best_model.predict(dss_input, verbose=0)
dss_inv   = scaler_y.inverse_transform(dss_preds[:, :1]).flatten()

plt.figure(figsize=(14, 4))
plt.plot(dss_inv, marker="o", markersize=4, color="#7c3aed", linewidth=1.5, label="3-Month Ahead Forecast")
plt.axhline(dss_inv.mean(), color="gray", linestyle="--", linewidth=1, label="Mean Forecast")
plt.fill_between(range(len(dss_inv)), dss_inv * 0.93, dss_inv * 1.07,
                 alpha=0.2, color="#7c3aed", label="±7% Confidence Band")
plt.title(f"RQ3 DSS — Rolling 3-Month Demand Forecast ({best_model_name})")
plt.xlabel("Forecast Window"); plt.ylabel("Quantity (MT)")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("results/rq3_dss_forecast.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:

display(df_rq1_results.round(3))

print(f"RF  Procurement Price  RMSE={rf_price_metrics['RMSE']:,.1f}  MAPE={rf_price_metrics['MAPE%']:.2f}%  R²={rf_price_metrics['R2']:.3f}")
display(df_storage.round(3))
print(f"GBM Procurement Sched  MAPE={gbm_mape:.2f}%  R²={gbm_r2:.3f}")
print(f"MLP Transport Cost     RMSE={mlp_rmse:.2f}   MAPE={mlp_mape:.2f}%")


print(f"RF  Risk Classifier    AUC-ROC={auc_roc:.3f}")
print(f"GBM Anomaly Detector   Precision={prec:.3f}  Recall={rec:.3f}  F1={f1:.3f}")
print(f"DSS Forecast Engine    Best model: {best_model_name}")
print("=" * 60)
